In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "india"
vehicle = "rice"

In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,intervention,174,0,5.114704
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,intervention,174,0,7.791002
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,intervention,174,0,4.519971
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,intervention,174,0,2.319459
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,intervention,174,0,2.259985
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,54,0,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,54,0,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,54,0,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,54,0,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

mild          270000
moderate      270000
not_anemic    270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  5.751678e+06
              2                  4.848211e+06
              3                  4.372048e+06
              4                  4.111569e+06
              5                  3.978572e+06
intervention  1                  5.751678e+06
              2                  4.848211e+06
              3                  4.372048e+06
              4                  4.111569e+06
              5                  3.978572e+06
zero          1                  5.751663e+06
              2                  4.848199e+06
              3                  4.372039e+06
              4                  4.111559e+06
              5                  3.978570e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  3.297270e+06
              2                  2.641252e+06
              3                  2.255347e+06
              4                  1.951432e+06
              5                  1.577134e+06
intervention  1                  3.297270e+06
              2                  2.641252e+06
              3                  2.255347e+06
              4                  1.951432e+06
              5                  1.577134e+06
zero          1                  3.428348e+06
              2                  2.732606e+06
              3                  2.324187e+06
              4                  2.008442e+06
              5                  1.599863e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.573271
              2                  0.544789
              3                  0.515856
              4                  0.474620
              5                  0.396407
intervention  1                  0.573271
              2                  0.544789
              3                  0.515856
              4                  0.474620
              5                  0.396407
zero          1                  0.596062
              2                  0.563633
              3                  0.531602
              4                  0.488487
              5                  0.402120
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,49649.700497
1,Female,0.0,0.019178,not_pregnant,2,43575.622144
2,Female,0.0,0.019178,not_pregnant,3,38689.356750
3,Female,0.0,0.019178,not_pregnant,4,36440.833935
4,Female,0.0,0.019178,not_pregnant,5,29202.585338
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,15176.692284
281,Male,95.0,125.000000,not_pregnant,2,15848.509818
282,Male,95.0,125.000000,not_pregnant,3,16370.176208
283,Male,95.0,125.000000,not_pregnant,4,17265.313670


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    5.312734e+06
2    4.479329e+06
3    4.040741e+06
4    3.799765e+06
5    3.668830e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  3.045637e+06
              2                  2.440289e+06
              3                  2.084440e+06
              4                  1.803444e+06
              5                  1.454350e+06
intervention  1                  3.045637e+06
              2                  2.440289e+06
              3                  2.084440e+06
              4                  1.803444e+06
              5                  1.454350e+06
zero          1                  3.166720e+06
              2                  2.524699e+06
              3                  2.148068e+06
              4                  1.856135e+06
              5                  1.475310e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,intervention,174,0,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,intervention,174,0,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,intervention,174,0,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,intervention,174,0,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,intervention,174,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,54,0,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,54,0,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,54,0,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,54,0,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['maternal_disorders_to_recovered_from_maternal_disorders',
       'no_transition',
       'susceptible_to_maternal_disorders_to_maternal_disorders'],
      dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  6.056562e+06
              2                  5.132795e+06
              3                  4.603061e+06
              4                  4.143960e+06
              5                  3.747214e+06
intervention  1                  6.056562e+06
              2                  5.132795e+06
              3                  4.603061e+06
              4                  4.143960e+06
              5                  3.747214e+06
zero          1                  6.122537e+06
              2                  5.177910e+06
              3                  4.635732e+06
              4                  4.171603e+06
              5                  3.758070e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,deaths,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,baseline,27,0,0.000000
1,deaths,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,baseline,27,0,0.000000
2,deaths,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,baseline,27,0,0.000000
3,deaths,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,baseline,27,0,0.000000
4,deaths,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,baseline,27,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
47995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,intervention,103,0,71.322406
47996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,intervention,103,0,52.716561
47997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,intervention,103,0,49.615587
47998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,intervention,103,0,21.706819


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  207427.264658
              2                  171483.872801
              3                  153442.404954
              4                  143913.111263
              5                  137695.658009
intervention  1                  207427.264658
              2                  171483.872801
              3                  153442.404954
              4                  143913.111263
              5                  137695.658009
zero          1                  208025.752677
              2                  171862.191653
              3                  153752.502373
              4                  144164.290172
              5                  137838.302821
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,48240.490460,zero
1,Female,0.0,0.019178,2,41704.307644,zero
2,Female,0.0,0.019178,3,36771.135988,zero
3,Female,0.0,0.019178,4,33854.718015,zero
4,Female,0.0,0.019178,5,26541.330813,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,11905.701977,intervention
746,Male,95.0,125.000000,2,12245.674227,intervention
747,Male,95.0,125.000000,3,12523.793023,intervention
748,Male,95.0,125.000000,4,13160.712426,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.276127e+08
              2                  1.186610e+08
              3                  1.163794e+08
              4                  1.101636e+08
              5                  1.035270e+08
intervention  1                  1.276127e+08
              2                  1.186610e+08
              3                  1.163794e+08
              4                  1.101636e+08
              5                  1.035270e+08
zero          1                  1.359974e+08
              2                  1.266546e+08
              3                  1.233426e+08
              4                  1.160203e+08
              5                  1.066586e+08
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.306583e+08
              2                  1.211013e+08
              3                  1.184638e+08
              4                  1.119670e+08
              5                  1.049814e+08
intervention  1                  1.306583e+08
              2                  1.211013e+08
              3                  1.184638e+08
              4                  1.119670e+08
              5                  1.049814e+08
zero          1                  1.391641e+08
              2                  1.291793e+08
              3                  1.254907e+08
              4                  1.178764e+08
              5                  1.081339e+08
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  4416.960543
              2                  3857.477219
              3                  3430.077399
              4                  3211.128079
              5                  2666.226621
baseline      1                  4099.443838
              2                  3621.430801
              3                  3251.193731
              4                  3065.715463
              5                  2610.941169
intervention  1                  2318.253524
              2                  2208.781602
              3                  2119.015017
              4                  2104.809997
              5                  2186.645282
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)